### 실습 문제

CLIP으로 **제로샷 이미지 분류**를 해보세요. 한 장의 이미지를 여러 후보 텍스트 라벨과 비교하여, 가장 잘 맞는 라벨을 예측합니다.

1. 분류할 이미지 한 장을 준비하세요. (예: `skimage` 샘플 이미지 `chelsea.png` - 고양이 사진)
2. 후보 라벨 리스트를 만드세요. 예: `["a cat", "a dog", "a horse", "a rocket", "a cup of coffee"]`
3. 이미지와 각 라벨을 인코딩하고 L2 정규화한 뒤, 코사인 유사도를 softmax 확률로 변환하여 **확률이 가장 높은 라벨**을 출력하세요.

### 테스트 입력 예시

- 고양이(chelsea) 이미지 → 예측 라벨: `a cat`

## 정답

In [ ]:
import os
import numpy as np
import torch
import clip
import skimage
from PIL import Image

# GPU 가 있으면 cuda, 없으면 cpu 로 동작
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

In [ ]:
# 1. 분류할 이미지 한 장 (skimage 샘플 chelsea - 고양이 사진)
image = Image.open(os.path.join(skimage.data_dir, "chelsea.png")).convert("RGB")
image_input = preprocess(image).unsqueeze(0).to(device)

# 2. 후보 라벨 준비
labels = ["a cat", "a dog", "a horse", "a rocket", "a cup of coffee"]
text_tokens = clip.tokenize(["This is " + label for label in labels]).to(device)

# 3. 인코딩 + L2 정규화 + 코사인 유사도(softmax 확률)
with torch.no_grad():
    image_features = model.encode_image(image_input).float()
    text_features = model.encode_text(text_tokens).float()

image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)

# 100.0 은 CLIP 에서 일반적으로 사용하는 스케일 계수
probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)[0].cpu().numpy()

# 확률이 높은 순으로 출력
for label, p in sorted(zip(labels, probs), key=lambda x: x[1], reverse=True):
    print(f"{p * 100:5.1f}%  {label}")

print(f"\n예측 라벨: {labels[int(probs.argmax())]}")